In [1]:
import import_ipynb

import os
import cv2
import mediapipe as mp
import pandas as pd
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
from scipy.signal import find_peaks

from geometry_angles import extract_geometry_features_per_rep
from duration import extract_temporal_features_per_rep
from velocity import extract_kinetic_features_per_rep
from stability import extract_stability_features_per_rep 

In [2]:
# Initialize the modern Pose Landmarker
model_path = "../pose_landmarker.task"

# Ensure the model file is present
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Please download '{model_path}' using the curl command before running.")

options = vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=model_path),
    running_mode=vision.RunningMode.VIDEO,
    num_poses=1
)

def extract_coordinates_from_video(video_path, label, landmarker):
    """
    Reads a video file, extracts Hip, Knee, Ankle, and Shoulder coordinates 
    for each frame using the modern Tasks API, and returns a list of rows.
    """
    cap = cv2.VideoCapture(video_path)
    video_name = os.path.basename(video_path)
    data_rows = []
    frame_count = 0

    print(f"Processing: {video_name}...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Convert the frame to a MediaPipe Image object
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # Calculate timestamp in milliseconds
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        timestamp_ms = int((frame_count / fps) * 1000)

        # Run inference
        results = landmarker.detect_for_video(mp_image, timestamp_ms)

        # Check if person was found
        if results.pose_landmarks:
            landmarks = results.pose_landmarks[0]

            # Landmark indices
            LEFT = {
                "shoulder": 11,
                "hip": 23,
                "knee": 25,
                "ankle": 27,
            }

            RIGHT = {
                "shoulder": 12,
                "hip": 24,
                "knee": 26,
                "ankle": 28,
            }

            # Compute average visibility for each side
            left_visibility = (
                landmarks[LEFT["shoulder"]].visibility +
                landmarks[LEFT["hip"]].visibility +
                landmarks[LEFT["knee"]].visibility +
                landmarks[LEFT["ankle"]].visibility
            ) / 4

            right_visibility = (
                landmarks[RIGHT["shoulder"]].visibility +
                landmarks[RIGHT["hip"]].visibility +
                landmarks[RIGHT["knee"]].visibility +
                landmarks[RIGHT["ankle"]].visibility
            ) / 4

            # Choose the side with better visibility
            if left_visibility >= right_visibility:
                side = LEFT
                body_side = "left"
                visibility = left_visibility
            else:
                side = RIGHT
                body_side = "right"
                visibility = right_visibility

            row = {
                "video_name": video_name,
                "frame": frame_count,
                "label": label,

                "body_side": body_side,
                "visibility": visibility,

                "shoulder_x": landmarks[side["shoulder"]].x,
                "shoulder_y": landmarks[side["shoulder"]].y,

                "hip_x": landmarks[side["hip"]].x,
                "hip_y": landmarks[side["hip"]].y,

                "knee_x": landmarks[side["knee"]].x,
                "knee_y": landmarks[side["knee"]].y,

                "ankle_x": landmarks[side["ankle"]].x,
                "ankle_y": landmarks[side["ankle"]].y,
            }
            data_rows.append(row)
        
        frame_count += 1

    cap.release()
    return data_rows

def create_dataset_from_videos():
    video_root = "../videos"
    output_folder = "../edited_datasets"
    output_csv = os.path.join(output_folder, "squat_features.csv")

    valid_extensions = (".mp4", ".avi", ".mov")

    # Create output folder if needed
    os.makedirs(output_folder, exist_ok=True)

    # Skip if CSV already exists
    if os.path.exists(output_csv):
        print(f"'{output_csv}' already exists. Skipping dataset creation.")
        return

    if not os.path.exists(video_root):
        print(f"Error: Folder '{video_root}' not found.")
        return

    all_extracted_data = []

    # Process both label folders
    folder_labels = {
        "good_form": "good",
        "bad_form": "bad"
    }

    for folder_name, label in folder_labels.items():

        folder_path = os.path.join(video_root, folder_name)

        if not os.path.isdir(folder_path):
            print(f"Warning: Folder '{folder_path}' not found. Skipping.")
            continue

        for file_name in os.listdir(folder_path):

            if not file_name.lower().endswith(valid_extensions):
                continue

            video_path = os.path.join(folder_path, file_name)

            # Create a fresh Pose Landmarker for each video
            with vision.PoseLandmarker.create_from_options(options) as landmarker:
                video_data = extract_coordinates_from_video(
                    video_path,
                    label,
                    landmarker
                )

            all_extracted_data.extend(video_data)

    if all_extracted_data:
        df = pd.DataFrame(all_extracted_data)
        df.to_csv(output_csv, index=False)
        print(f"\nSuccess! Dataset saved to '{output_csv}' (Shape: {df.shape})")
    else:
        print("No data extracted.")

In [3]:
create_dataset_from_videos()

Processing: good_form_no_bar.mp4...
Processing: good_no_weights_personal.mp4...
Processing: good_squad_video.mp4...
Processing: good_squat_short.mp4...
Processing: good_weights_personal.mp4...
Processing: bad_bended_back.mp4...
Processing: bad_get_up_leaning.mp4...
Processing: bad_half_reps_no_weights.mp4...
Processing: bad_half_reps_weights.mp4...

Success! Dataset saved to '../edited_datasets\squat_features.csv' (Shape: (8308, 13))


In [4]:
INPUT_CSV = "../edited_datasets/squat_features.csv"
OUTPUT_CSV = "../edited_datasets/squat_features_engineered.csv"

SMOOTHING_WINDOW = 5
PEAK_PROMINENCE = 0.05
PEAK_DISTANCE = 20


def load_data(csv_path):
    """Load and sort pose data."""
    df = pd.read_csv(csv_path)
    df = df.sort_values(["video_name", "frame"]).reset_index(drop=True)
    return df


def add_smoothed_features(df, window=SMOOTHING_WINDOW):
    """Smooth landmark coordinates using rolling mean (reduces jitter)."""

    joints = ["shoulder", "hip", "knee", "ankle"]

    for joint in joints:
        for axis in ["x", "y"]:
            col = f"{joint}_{axis}"

            df[f"{col}_smooth"] = (
                df.groupby("video_name")[col]
                .transform(
                    lambda x: x.rolling(window=window, min_periods=1).mean()
                )
            )

    return df


def add_velocity_from_smoothed(df):
    """Compute velocity using smoothed coordinates (IMPORTANT FIX)."""

    joints = ["shoulder", "hip", "knee", "ankle"]

    for joint in joints:
        for axis in ["x", "y"]:
            smooth_col = f"{joint}_{axis}_smooth"

            df[f"{joint}_velocity_{axis}"] = (
                df.groupby("video_name")[smooth_col]
                .diff()
                .fillna(0)
            )

    # Useful derived metric
    df["hip_speed"] = df["hip_velocity_y"].abs()

    return df


def add_relative_position_features(df):
    """Body structure relationships (frame-based, no change needed)."""

    df["hip_to_knee_y"] = df["hip_y"] - df["knee_y"]
    df["shoulder_to_hip_y"] = df["shoulder_y"] - df["hip_y"]

    return df


def save_data(df, output_path):
    """Save final dataset."""
    df.to_csv(output_path, index=False)
    print(f"Saved engineered features to '{output_path}'")
    print(f"Shape: {df.shape}")

def add_repetitions_column(df, prominence=PEAK_PROMINENCE, distance=PEAK_DISTANCE):
    """
    Automatically segments and numbers individual squat repetitions per video
    """
    df['rep_number'] = 0
    unique_videos = df['video_name'].unique()

    for video_name in unique_videos:
        video_indices = df[df['video_name'] == video_name].index
        video_df = df.loc[video_indices].sort_values('frame')
        
        signal = video_df['hip_y_smooth'].values
        bottom_indices, _ = find_peaks(signal, prominence=prominence, distance=distance)
        
        if len(bottom_indices) == 0:
            continue
            
        for i, bottom_idx in enumerate(bottom_indices):
            if i == 0:
                start_idx = np.argmin(signal[:bottom_idx])
            else:
                prev_bottom = bottom_indices[i-1]
                start_idx = prev_bottom + np.argmin(signal[prev_bottom:bottom_idx])
                
            if i == len(bottom_indices) - 1:
                end_idx = bottom_idx + np.argmin(signal[bottom_idx:])
            else:
                next_bottom = bottom_indices[i+1]
                end_idx = bottom_idx + np.argmin(signal[bottom_idx:next_bottom])
            
            actual_row_start = video_df.index[start_idx]
            actual_row_end = video_df.index[end_idx]
            
            df.loc[actual_row_start:actual_row_end, 'rep_number'] = i + 1

    return df


def create_engineered_dataset():
    df = load_data(INPUT_CSV)

    df = add_smoothed_features(df)
    df = add_repetitions_column(df)
    df = add_velocity_from_smoothed(df)
    df = add_relative_position_features(df)

    save_data(df, OUTPUT_CSV)

    return df

In [5]:
df = create_engineered_dataset()

Saved engineered features to '../edited_datasets/squat_features_engineered.csv'
Shape: (8308, 33)


In [6]:
def build_vector_dataset(input_csv_path, output_csv_path, fps=30):
    """
    Groups the frame-by-frame data by video and rep number, extracts high-level 
    biomechanical feature vectors, and exports a final tabular dataset for ML.
    """
    # 1. Load the dataset that contains your 'rep_number' tags
    df = pd.read_csv(input_csv_path)
    
    final_rows = []
    
    # 2. Group data by video to handle fatigue baseline tracking per set
    unique_videos = df['video_name'].unique()
    
    for video in unique_videos:
        video_df = df[df['video_name'] == video]
        
        # Find the average ascent velocity of Rep 1 for this video baseline
        # This is needed to calculate the velocity loss context feature.
        rep1_df = video_df[video_df['rep_number'] == 1]
        if len(rep1_df) > 0:
            bottom_idx_rep1 = rep1_df['hip_y_smooth'].idxmax()
            ascent_rep1 = rep1_df.loc[bottom_idx_rep1:]
            rep_1_avg_ascent_vel = abs(ascent_rep1['hip_velocity_y'].mean()) if len(ascent_rep1) > 0 else None
        else:
            rep_1_avg_ascent_vel = None

        # 3. Group by repetition within this video
        rep_groups = video_df.groupby('rep_number')
        
        for rep_num, rep_df in rep_groups:
            # Ignore frames where no active repetition is occurring
            if rep_num == 0:
                continue
                
            # Run all 4 extractor sub-routines to generate feature chunks
            try:
                geo_feats = extract_geometry_features_per_rep(rep_df)
                temp_feats = extract_temporal_features_per_rep(rep_df, fps=fps)
                stab_feats = extract_stability_features_per_rep(rep_df)
                kin_feats = extract_kinetic_features_per_rep(rep_df, rep_1_avg_ascent_vel=rep_1_avg_ascent_vel)
                
                # 4. Merge metadata, target labels, and all feature dictionaries together
                master_rep_vector = {
                    'video_name': video,
                    'rep_number': rep_num,
                    'label': rep_df['label'].iloc[0],  # Target class: 'good' or 'bad'
                    'body_side': rep_df['body_side'].iloc[0],
                    **geo_feats,
                    **temp_feats,
                    **stab_feats,
                    **kin_feats
                }
                
                final_rows.append(master_rep_vector)
                
            except Exception as e:
                print(f"Skipping video {video} Rep {rep_num} due to processing error: {e}")
                continue

    # 5. Convert lists of vectors directly into a standard tabular DataFrame
    vector_df = pd.DataFrame(final_rows)
    
    # Save to your local file system
    vector_df.to_csv(output_csv_path, index=False)
    
    print("\n--- Dataset Assembly Complete ---")
    print(f"Successfully processed {len(vector_df)} total individual repetitions.")
    print(f"Final Data Matrix Dimensions: {vector_df.shape}")
    print(f"File stored safely at: '{output_csv_path}'")
    
    return vector_df

In [7]:
INPUT_FILE = "../edited_datasets/squat_features_engineered.csv" 
OUTPUT_FILE = "../edited_datasets/squat_rep_vectors_ml.csv"

build_vector_dataset(INPUT_FILE, OUTPUT_FILE)


--- Dataset Assembly Complete ---
Successfully processed 34 total individual repetitions.
Final Data Matrix Dimensions: (34, 35)
File stored safely at: '../edited_datasets/squat_rep_vectors_ml.csv'


,video_name,rep_number,label,body_side,min_knee_angle,min_hip_angle,max_torso_lean_degrees,max_depth_ratio,hip_to_ankle_bottom_alignment,avg_knee_angle_descent,...,torso_lean_to_depth_ratio,knee_travel_to_depth_ratio,ascent_path_symmetry,avg_descent_velocity_y,avg_ascent_velocity_y,max_acceleration_y,max_jerk_y,peak_ascent_velocity_y,sticking_point_velocity_drop,velocity_loss_vs_rep_1
0,bad_bended_back.mp4,1,bad,right,43.049851,14.975019,68.141123,0.197063,0.117924,84.434524,...,1.006064,0.374258,0.965464,0.001505,0.002094,0.000940,0.001317,0.004549,0.004409,1.000000
1,bad_bended_back.mp4,2,bad,right,63.280149,14.802451,75.545162,0.287743,0.152826,105.864161,...,1.399116,0.325147,0.962484,0.000744,0.001338,0.000903,0.001620,0.003853,0.003834,0.638720
2,bad_bended_back.mp4,3,bad,right,61.964157,16.396533,74.853119,0.314073,0.155097,100.319773,...,1.643166,0.549285,0.992769,0.000877,0.001790,0.000951,0.001402,0.004343,0.004332,0.854539
3,bad_get_up_leaning.mp4,1,bad,right,38.295569,18.269700,87.335510,0.191892,0.098779,88.367445,...,0.906519,0.411937,0.059349,0.001734,0.002132,0.001331,0.001307,0.004931,0.004247,1.000000
4,bad_get_up_leaning.mp4,2,bad,right,38.105101,13.484594,80.203003,0.146324,0.106704,98.216146,...,0.834887,0.437577,0.543970,0.001079,0.001492,0.001225,0.001723,0.003409,0.003375,0.699875
5,bad_get_up_leaning.mp4,3,bad,right,35.678254,12.922544,83.277382,0.152909,0.101569,75.276638,...,0.953147,0.493887,-0.095223,0.001267,0.002171,0.001102,0.001329,0.003721,0.003517,1.018070
6,bad_half_reps_no_weights.mp4,1,bad,right,47.033678,53.148884,49.199953,0.348653,0.081965,106.175811,...,1.023032,0.728665,0.836469,0.000889,0.001936,0.001049,0.001376,0.003043,0.002637,1.000000
7,bad_half_reps_no_weights.mp4,2,bad,right,53.120890,48.664004,51.810001,0.365088,0.101885,108.407434,...,1.203982,0.742951,0.988062,0.000758,0.000893,0.000712,0.000999,0.003046,0.003045,0.461061
8,bad_half_reps_no_weights.mp4,3,bad,right,41.808997,44.042619,52.626783,0.298814,0.082155,86.963698,...,1.050683,0.693808,0.865817,0.001061,0.001165,0.000818,0.001168,0.003378,0.003374,0.601871
9,bad_half_reps_no_weights.mp4,4,bad,right,45.024979,50.241355,50.645876,0.336651,0.075493,104.616632,...,1.083447,0.827908,0.984034,0.001080,0.001052,0.000818,0.001022,0.003152,0.003127,0.543048
